# Batching Example

This notebook demonstrates how to create a batch request, track its status, and download the resulting output file with the GigaChat API.


Import required libraries for working with batch files and the GigaChat client.


In [ ]:
import base64
import getpass
import os

from gigachat import GigaChat


Set up GigaChat credentials and scope. The input will be hidden for security.


In [ ]:
if "GIGACHAT_CREDENTIALS" not in os.environ:
    os.environ["GIGACHAT_CREDENTIALS"] = getpass.getpass("GigaChat Credentials: ")

if "GIGACHAT_SCOPE" not in os.environ:
    os.environ["GIGACHAT_SCOPE"] = getpass.getpass("GigaChat Scope: ")


## Load Batch Input File

Read a JSONL file with batch requests. The API currently supports up to 500 lines per file.


In [ ]:
with open("batch_file_500.jsonl", "rb") as f:
    data = f.read()


## Create Batch

Create a batch using the appropriate method for your file, such as `chat_completions` or `embedder`.


In [ ]:
with GigaChat(verify_ssl_certs=False) as giga:
    batch_creation = giga.create_batch(data, method="chat_completions")
    print(batch_creation)


## Check Batch Status

Batch jobs move through `created`, `in_progress`, and `completed` states. Use `get_batches` to poll the current status.


In [ ]:
with GigaChat(verify_ssl_certs=False) as giga:
    result = giga.get_batches(batch_id=batch_creation.id_)
    print(result)


## Download And Decode Output

After the batch completes, use the `output_file_id` field to fetch the output file content, decode the base64 payload, and save it as a JSONL file.


In [ ]:
with GigaChat(verify_ssl_certs=False) as giga:
    output_file = giga.get_file_content(result.output_file_id)

decoded = base64.b64decode(output_file.content)

with open("batch_answers_500.jsonl", "wb") as f:
    f.write(decoded)
